# white_hetero

> 对应代码：`EconometricsCode/code_in_notes/Chap.9/white_hetero.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.9`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.9")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

定义模拟程序 dgp：选项 obs 为样本量（默认 100），robust 为开关（出现时使用稳健标准误），b 为真实系数。

构造异方差数据：x 为标准正态，误差标准差 σ=|x|，于是 y = b·x + σ·ε 的误差方差随 |x| 增大而增大——这是典型的异方差结构，且方差是解释变量的函数，正对应 White 标准误所处理的情形。

做 OLS 回归：`robust' 宏为空时使用经典标准误，传入 robust 选项时则使用 White/Huber 异方差稳健标准误。返回系数与标准误。

用 1.96 为临界值做双侧 t 检验：t 统计量绝对值超过 1.96 记为拒绝原假设（rejected=1），否则为 0。

In [ ]:
%%stata
cap program drop dgp
program define dgp, rclass
	syntax [, obs(integer 100) robust b(real 0)]
	drop _all
	set obs `obs'
	tempvar x y sigma
	gen `x'=rnormal()
	gen `sigma'=sqrt(`x'^2)
	gen `y'=`b'*`x'+`sigma'*rnormal()
	quietly: reg `y' `x', `robust'
	return scalar b=_b[`x']
	return scalar se=_se[`x']
	if abs(_b[`x']/_se[`x'])>=1.96 {
		return scalar rejected=1
	}
	else {
		return scalar rejected=0
	}
end

第一组模拟：在 b=0（原假设成立）且存在异方差的情形下，使用经典（同方差）标准误重复 1000 次。拒绝频率即检验的实际水平——由于经典标准误在异方差下是错误的，实际水平会明显偏离名义水平 5%，即检验失真（size distortion）。

In [ ]:
%%stata
simulate rejected=r(rejected) b=r(b) se=r(se) ,reps(1000):dgp
su

第二组模拟：同样的异方差 DGP，但改用 White 稳健标准误。拒绝频率回到 5% 附近，说明 White 标准误在异方差下给出了渐近正确的推断，检验的实际水平得到校正。

In [ ]:
%%stata
simulate rejected=r(rejected) b=r(b) se=r(se) ,reps(1000):dgp, robust
su